<a href="https://colab.research.google.com/github/Arma3071/Flyrank-AI-notebooks/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN')")

# List top-level subsets/folders in the repo
con.sql("SELECT * FROM glob('hf://datasets/FlyRank/internship-warehouse/**')").show()

┌────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                  file                                                  │
│                                                varchar                                                 │
├────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ hf://datasets/FlyRank/internship-warehouse/.gitattributes                                              │
│ hf://datasets/FlyRank/internship-warehouse/README.md                                                   │
│ hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet                                         │
│ hf://datasets/FlyRank/internship-warehouse/dim_content.parquet                                         │
│ hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet │
│ hf://datasets/FlyRank/internship-wa

In [2]:
for t in ["dim_clients", "dim_content", "fact_content_query_90d"]:
    print(f"--- {t} ---")
    con.sql(f"DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/{t}.parquet')").show()

print("--- fact_content_daily_performance (sample) ---")
con.sql("DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')").show()

--- dim_clients ---
┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ is_active           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ has_gsc_access      │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ has_ga4_access      │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ access_profile      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_created_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_updated_date │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_start      │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_start      │ DATE        │ YES   

In [13]:
import pandas as pd

# --- base table ---

BASE = "hf://datasets/FlyRank/internship-warehouse"

# --- content-level attributes ---
con.sql(f"""
CREATE OR REPLACE TABLE content_attrs AS
SELECT
    content_hash_id,
    ANY_VALUE(client_hash_id)      AS client_hash_id,
    MAX(content_updated_date)      AS content_updated_date,
    MAX(last_optimized_date)       AS last_optimized_date,
    ANY_VALUE(content_type)        AS content_type,
    ANY_VALUE(word_count)          AS word_count,
    BOOL_OR(is_published)          AS is_published,
    BOOL_OR(is_deleted)            AS is_deleted
FROM read_parquet('{BASE}/dim_content.parquet')
GROUP BY content_hash_id
""")

# --- content-level performance ---
con.sql(f"""
CREATE OR REPLACE TABLE content_perf AS
SELECT
    content_hash_id,
    ANY_VALUE(client_hash_id) AS client_hash_id,
    MAX(window_end)           AS window_end,
    SUM(impressions_90d)      AS impressions_90d,
    SUM(clicks_90d)           AS clicks_90d,
    SUM(impressions_last30)   AS impressions_last30,
    SUM(clicks_last30)        AS clicks_last30,
    SUM(impressions_prev30)   AS impressions_prev30,
    SUM(clicks_prev30)        AS clicks_prev30,
    SUM(avg_position_90d * impressions_90d) / NULLIF(SUM(impressions_90d),0)     AS avg_position_90d,
    SUM(avg_position_last30 * impressions_last30) / NULLIF(SUM(impressions_last30),0) AS avg_position_last30,
    SUM(avg_position_prev30 * impressions_prev30) / NULLIF(SUM(impressions_prev30),0) AS avg_position_prev30
FROM read_parquet('{BASE}/fact_content_query_90d.parquet')
GROUP BY content_hash_id
""")

# --- content-level anonymization/rare-query noise ---
con.sql(f"""
CREATE OR REPLACE TABLE content_anon AS
SELECT
    content_hash_id,
    SUM(rare_query_count) AS rare_query_count,
    SUM(rare_impressions_share * impressions_90d) / NULLIF(SUM(impressions_90d),0)       AS rare_impressions_share,
    SUM(anonymized_impressions_share * impressions_90d) / NULLIF(SUM(impressions_90d),0) AS anonymized_impressions_share
FROM read_parquet('{BASE}/fact_content_query_90d.parquet')
GROUP BY content_hash_id
""")

base = con.sql("""
SELECT p.*, a.content_updated_date, a.last_optimized_date, a.content_type,
       a.word_count, a.is_published, a.is_deleted,
       an.rare_impressions_share, an.anonymized_impressions_share
FROM content_perf p
JOIN content_attrs a USING (content_hash_id)
JOIN content_anon an USING (content_hash_id)
WHERE a.is_published = TRUE AND a.is_deleted = FALSE
""").df()

print("rows before any filter:", len(base))

# as-of date: latest window_end in the data
as_of_date = base["window_end"].max()
print("as_of_date:", as_of_date)

# --- filter 1: anonymization/rare-query noise ---
ANON_THRESHOLD = 0.30
before_anon = len(base)
base = base[
    (base["anonymized_impressions_share"].fillna(0) <= ANON_THRESHOLD) &
    (base["rare_impressions_share"].fillna(0) <= ANON_THRESHOLD)
].copy()
print(f"rows dropped for high anonymized/rare share (>{ANON_THRESHOLD}): {before_anon - len(base)}")

# --- filter 2: low-impression rows ---
before_impr = len(base)
base = base[base["impressions_90d"] >= 10].copy()
print(f"rows dropped for impressions_90d<10: {before_impr - len(base)}")

# --- derived fields ---
base["days_since_update"] = (pd.to_datetime(as_of_date) - pd.to_datetime(base["content_updated_date"])).dt.days
base["ctr_90d"]    = base["clicks_90d"] / base["impressions_90d"]
base["ctr_last30"] = base["clicks_last30"] / base["impressions_last30"].replace(0, pd.NA)
base["ctr_prev30"] = base["clicks_prev30"] / base["impressions_prev30"].replace(0, pd.NA)
base["ctr_trend"]  = base["ctr_last30"] - base["ctr_prev30"]

# --- filter 3: future-dated content_updated_date (leak guard) ---
before_future = len(base)
base = base[base["days_since_update"] >= 0].copy()
print(f"rows dropped for content_updated_date after as_of_date: {before_future - len(base)}")

print("rows after all filters:", len(base))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows before any filter: 133801
as_of_date: 2026-06-30 00:00:00
rows dropped for high anonymized/rare share (>0.3): 125730
rows dropped for impressions_90d<10: 0
rows dropped for content_updated_date after as_of_date: 978
rows after all filters: 7093


In [14]:
bins = [-1, 30, 90, 180, 365, 10_000]
labels = ["<30d", "30-90d", "90-180d", "180-365d", "365d+"]
base["staleness_bucket"] = pd.cut(base["days_since_update"], bins=bins, labels=labels)

staleness_table = (
    base.groupby("staleness_bucket", observed=True)
        .agg(n=("content_hash_id", "count"),
             mean_ctr_trend=("ctr_trend", "mean"),
             mean_ctr_90d=("ctr_90d", "mean"))
        .reset_index()
)
print(staleness_table)
#MIXED

  staleness_bucket     n mean_ctr_trend  mean_ctr_90d
0             <30d  1352      -0.000263      0.000413
1           30-90d  4842      -0.000042      0.000282
2          90-180d   870      -0.000076      0.000295
3         180-365d    29      -0.004882      0.004463


In [15]:
pos_bins = [0, 3, 10, 20, 50, 1000]
pos_labels = ["1-3", "4-10", "11-20", "21-50", "51+"]
base["position_bucket"] = pd.cut(base["avg_position_90d"], bins=pos_bins, labels=pos_labels)

position_table = (
    base.groupby("position_bucket", observed=True)
        .agg(n=("content_hash_id", "count"),
             mean_ctr_90d=("ctr_90d", "mean"))
        .reset_index()
)
print(position_table)

# sanity check on the surprisingly low CTR values before finalizing the verdict
raw_check = con.sql(f"""
SELECT content_hash_id, query_hash_id, clicks_90d, impressions_90d,
       clicks_90d::DOUBLE / NULLIF(impressions_90d,0) AS row_ctr
FROM read_parquet('{BASE}/fact_content_query_90d.parquet')
ORDER BY impressions_90d DESC
LIMIT 15
""").df()
print(raw_check)
#raw_check:MIXED

  position_bucket     n  mean_ctr_90d
0             1-3   285      0.000478
1            4-10   908      0.001106
2           11-20   479      0.000692
3           21-50  1649      0.000267
4             51+  3747      0.000106


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

             content_hash_id           query_hash_id  clicks_90d  \
0   content_943dc881428182b8  query_1e12d78d0219e482          55   
1   content_11bf4c33adea7bdc  query_c3e1dca2228f7a00           0   
2   content_d0acf7062bc6b257  query_1e8e533759e5af65           0   
3   content_c60628276389acbb  query_c192c1192eb8048c           0   
4   content_99fc6465edb0e52c  query_5913910fecd6014c           0   
5   content_39e19a3ec2d95f9d  query_395ba828a7f68324           0   
6   content_987d251ee617d9c6  query_ab81171134a428bb         783   
7   content_32c5cc913fb4ff41  query_3cf4167de6c7be93           2   
8   content_012de75c008aa653  query_76ae356cb67f276f           0   
9   content_7471467133493ce6  query_1e8e533759e5af65           0   
10  content_23a42776a7009b65  query_f46ad60559dfccea           0   
11  content_44f34c0a90047651  query_019864b9db9390a9           2   
12  content_bddfdd871aa09fbe  query_019864b9db9390a9           1   
13  content_8e1334d6356668e3  query_d6fdb877bb44

In [16]:
expected = position_table.set_index("position_bucket")["mean_ctr_90d"].to_dict()
base["expected_ctr"] = base["position_bucket"].map(expected).astype(float)
base["ctr_gap"] = base["expected_ctr"] - base["ctr_90d"]

base["reason_code"] = "CTR_BELOW_EXPECTED_FOR_POSITION"
base["score"] = (base["ctr_gap"].clip(lower=0) * base["impressions_90d"]).round(2)
base["action"] = base["score"].apply(lambda s: "OPTIMIZE_CTR" if s > 0 else "MONITOR")

queue = base.sort_values("score", ascending=False)[[
    "content_hash_id", "client_hash_id", "score", "reason_code", "action",
    "avg_position_90d", "ctr_90d", "expected_ctr", "impressions_90d", "days_since_update"
]]

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(queue.head(10))

                 content_hash_id           client_hash_id   score  \
50080   content_39e19a3ec2d95f9d  client_1a730cb2640a1abf  419.33   
56907   content_11bf4c33adea7bdc  client_0fa64a184f18a4a0  366.45   
38833   content_c1f764a2f362d1c3  client_a80fca3f171ed1de  334.94   
103014  content_012de75c008aa653  client_a80fca3f171ed1de  280.40   
70991   content_9540d884af3e41fd  client_a80fca3f171ed1de  269.07   
17529   content_65c75874a23fca87  client_23a62021009f63c4  236.92   
60892   content_23a42776a7009b65  client_73cda7b4e4f265ea  225.10   
70455   content_b902320872acab45  client_8ddc46da5414ffd8  208.14   
113885  content_44f34c0a90047651  client_23a62021009f63c4  194.94   
39646   content_99fc6465edb0e52c  client_cd12bcfd98942aa1  192.04   

                            reason_code        action  avg_position_90d  \
50080   CTR_BELOW_EXPECTED_FOR_POSITION  OPTIMIZE_CTR          9.789889   
56907   CTR_BELOW_EXPECTED_FOR_POSITION  OPTIMIZE_CTR          8.395685   
38833   CTR_BEL

In [17]:
top10 = queue.head(10).copy()

print(f"{'content_hash_id':<28} {'action':<14} {'score':>10}  why / what would make it wrong")
print("-" * 110)

for i, row in top10.iterrows():
    why = (f"pos={row['avg_position_90d']:.1f}, ctr={row['ctr_90d']:.4f} vs "
           f"expected={row['expected_ctr']:.4f}, impr_90d={row['impressions_90d']:.0f}, "
           f"days_since_update={row['days_since_update']:.0f}")

    if row["days_since_update"] <= 7:
        wrong_if = "wrong if: content was JUST updated — CTR hasn't had time to respond yet"
    elif row["impressions_90d"] > 300_000:
        wrong_if = "wrong if: high-impression page is absorbing clicks via AI Overviews/answer boxes, not fixable via CTR tweaks"
    elif row["ctr_90d"] < 0.001:
        wrong_if = "wrong if: near-zero CTR reflects a tracking/parsing issue rather than a real ranking problem"
    else:
        wrong_if = "wrong if: expected_ctr benchmark for this position bucket has low n and isn't a reliable target"

    print(f"{row['content_hash_id']:<28} {row['action']:<14} {row['score']:>10.2f}  {why}")
    print(f"{'':<28} {'':<14} {'':>10}  {wrong_if}\n")

content_hash_id              action              score  why / what would make it wrong
--------------------------------------------------------------------------------------------------------------
content_39e19a3ec2d95f9d     OPTIMIZE_CTR       419.33  pos=9.8, ctr=0.0000 vs expected=0.0011, impr_90d=379229, days_since_update=36
                                                        wrong if: high-impression page is absorbing clicks via AI Overviews/answer boxes, not fixable via CTR tweaks

content_11bf4c33adea7bdc     OPTIMIZE_CTR       366.45  pos=8.4, ctr=0.0000 vs expected=0.0011, impr_90d=332312, days_since_update=41
                                                        wrong if: high-impression page is absorbing clicks via AI Overviews/answer boxes, not fixable via CTR tweaks

content_c1f764a2f362d1c3     OPTIMIZE_CTR       334.94  pos=7.3, ctr=0.0002 vs expected=0.0011, impr_90d=352652, days_since_update=41
                                                        wrong if: hi

Weak-score rows (bottom quartile of positive scores): 1068
                 content_hash_id  score  avg_position_90d  ctr_90d  \
78985   content_fd106cd1407f5f19   0.01         67.226190      0.0   
133441  content_919be6c449100c85   0.01         61.022901      0.0   
78953   content_fbf22f86b6011c15   0.01         75.838235      0.0   
1180    content_1b1f69effd0e4531   0.01         55.830645      0.0   
1175    content_1993d3491a5df75e   0.01         78.861538      0.0   
52810   content_1bf23eea87abe823   0.01         83.776119      0.0   
133333  content_7a96e12acd24a15e   0.01         52.698630      0.0   
1415    content_5b486675b8b5480e   0.01         77.522936      0.0   
1412    content_5ae42966d31d0deb   0.01         68.298246      0.0   
31135   content_1e8ec662d8ce771a   0.01         53.018519      0.0   

        impressions_90d  days_since_update  
78985              84.0                 41  
133441            131.0                125  
78953              68.0            

In [20]:
# 1. No future-window inputs
future_check = base[pd.to_datetime(base["content_updated_date"]) > pd.to_datetime(as_of_date)]
print(f"content_updated_date after as_of_date ({as_of_date}): {len(future_check)} rows — should be 0")
assert len(future_check) == 0, "Future-dated rows still present"

print(f"Future-leak filter dropped {before_future - len(base) if False else 'see Section 1 print'} rows (see Section 1 log)")

# 2. Anonymization filter record
print(f"Anonymization filter: dropped rows with anonymized/rare share > {ANON_THRESHOLD} (see Section 1 log)")

# 3. No label-derived inputs
print("\nColumns feeding the rule:", ["expected_ctr", "ctr_90d", "ctr_gap", "impressions_90d"])
print("All trailing-window inputs as of", as_of_date, "— none derived from outcomes/labels")

# 4. n printed for both bucket tables
print("\nStaleness bucket n's:\n", staleness_table[["staleness_bucket", "n"]])
print("\nPosition bucket n's:\n", position_table[["position_bucket", "n"]])

# 5. CSV regenerates cleanly
assert os.path.exists("work/outputs/baseline_action_score.csv"), "CSV missing — rerun Section 2"
recheck = pd.read_csv("work/outputs/baseline_action_score.csv")
print(f"\nCSV rows: {len(recheck)}, columns: {list(recheck.columns)}")

# 6. Known limitation
print("\nKnown limitation: CTR-vs-position signal is MIXED, not cleanly CONFIRMED. "
      "The relationship holds from position 4-10 through 51+ (CTR falls monotonically: "
      "0.0011 -> 0.0007 -> 0.0003 -> 0.0001), but position 1-3 breaks the pattern with "
      "lower CTR (0.000478) than 4-10 (0.001106). Row-level inspection of the raw fact table "
      "confirms this isn't a join/aggregation artifact -- many high-impression rows "
      "(150k-540k impressions) show zero clicks even at good positions, consistent with "
      "SERP-feature cannibalization (AI Overviews, answer boxes, featured snippets) "
      "absorbing the click without a visit. This makes expected_ctr for the 1-3 bucket "
      "an unreliable benchmark: it currently sits BELOW the 4-10 benchmark, which could "
      "mask real CTR problems on the site's best-ranked pages. Staleness signal is FALSE/weak-MIXED: "
      "no clean age-decay pattern among buckets with adequate n (<30d, 30-90d, 90-180d); "
      "the one bucket that looked confirming (180-365d, trend=-0.0049) has n=29, too small to trust.")

content_updated_date after as_of_date (2026-06-30 00:00:00): 0 rows — should be 0
Future-leak filter dropped see Section 1 print rows (see Section 1 log)
Anonymization filter: dropped rows with anonymized/rare share > 0.3 (see Section 1 log)

Columns feeding the rule: ['expected_ctr', 'ctr_90d', 'ctr_gap', 'impressions_90d']
All trailing-window inputs as of 2026-06-30 00:00:00 — none derived from outcomes/labels

Staleness bucket n's:
   staleness_bucket     n
0             <30d  1352
1           30-90d  4842
2          90-180d   870
3         180-365d    29

Position bucket n's:
   position_bucket     n
0             1-3   285
1            4-10   908
2           11-20   479
3           21-50  1649
4             51+  3747

CSV rows: 7093, columns: ['content_hash_id', 'client_hash_id', 'score', 'reason_code', 'action', 'avg_position_90d', 'ctr_90d', 'expected_ctr', 'impressions_90d', 'days_since_update']

Known limitation: CTR-vs-position signal is MIXED, not cleanly CONFIRMED. The rela